# CUDA FFT Performance Evaluation

Before running this notebook, select **Runtime > Change runtime type > T4 GPU** in Google Colab.

This notebook runs each CUDA block size once and generates `results.txt`, `execution_time.png`, and `speedup.png`.

In [ ]:
!nvidia-smi
!nvcc --version

Upload `fft_cuda.cu` from the project.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!nvcc -O3 fft_cuda.cu -o fft_cuda

In [ ]:
import re
import subprocess
import matplotlib.pyplot as plt

block_sizes = [32, 64, 128, 256, 512, 1024]
time_pattern = re.compile(r"completed in ([0-9.]+) seconds")
results = []

for block_size in block_sizes:
    output = subprocess.check_output(
        ["./fft_cuda", str(block_size)], text=True
    )
    match = time_pattern.search(output)
    if not match:
        raise RuntimeError(f"Could not read execution time:\n{output}")

    execution_time = float(match.group(1))
    results.append((block_size, execution_time))
    print(f"Block size {block_size}: {execution_time:.6f} seconds")

In [ ]:
baseline = results[0][1]

lines = [
    "CUDA Performance Results",
    "========================",
    "Dataset size: 1,048,576 samples",
    "Each configuration was run once.",
    "",
    f"{'Block size':<16} {'Time (seconds)':>16} {'Speedup':>12}",
    "-" * 46,
]

for block_size, execution_time in results:
    lines.append(
        f"{block_size:<16} {execution_time:>16.6f} "
        f"{baseline / execution_time:>12.4f}"
    )

result_text = "\n".join(lines) + "\n"
with open("results.txt", "w") as file:
    file.write(result_text)

print(result_text)

In [ ]:
block_sizes = [item[0] for item in results]
execution_times = [item[1] for item in results]
speedups = [baseline / execution_time for execution_time in execution_times]

plt.figure(figsize=(8, 5))
plt.plot(block_sizes, execution_times, marker="o", linewidth=2)
plt.title("CUDA: Threads per Block vs Execution Time")
plt.xlabel("Threads per block")
plt.ylabel("Execution time (seconds)")
plt.xticks(block_sizes)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("execution_time.png", dpi=200)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(block_sizes, speedups, marker="o", linewidth=2)
plt.title("CUDA: Threads per Block vs Speedup")
plt.xlabel("Threads per block")
plt.ylabel("Speedup")
plt.xticks(block_sizes)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("speedup.png", dpi=200)
plt.show()

In [ ]:
from google.colab import files

files.download("results.txt")
files.download("execution_time.png")
files.download("speedup.png")